# Analyse scientifique de la plateforme d'analyse de la criminalité urbaine

Ce notebook complète la plateforme technique afin de répondre aux observations de l'évaluation scientifique.

## Question de recherche proposée

**Comment les méthodes d'analyse de données et de Machine Learning peuvent-elles être utilisées pour extraire des connaissances pertinentes à partir de données criminelles agrégées malgré leurs limitations ?**

## Démarche

La structure suit l'esprit de **CRISP-DM** :

1. Compréhension du problème
2. Compréhension des données
3. Préparation des données
4. Analyse exploratoire (EDA)
5. Modélisation
6. Validation
7. Interprétation et discussion critique

> Important : les données sont agrégées et ne contiennent ni géolocalisation fine ni temporalité mensuelle. Les résultats restent exploratoires et ne doivent pas être interprétés comme une prédiction opérationnelle de la criminalité.

## 1. Imports et localisation du projet

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)
from sklearn.model_selection import (
    KFold,
    GroupKFold,
    cross_validate,
    cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd() / "criminalite_intelligente_v2",
        Path.cwd().parent / "criminalite_intelligente_v2",
    ]
    for candidate in candidates:
        if (candidate / "data" / "processed" / "infractions_clean.csv").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Projet introuvable. Lancez ce notebook depuis le dossier du projet "
        "ou placez-le dans/à côté de criminalite_intelligente_v2."
    )

ROOT = find_project_root()
print("Projet :", ROOT)

## 2. Chargement des données

On utilise à la fois les données brutes et nettoyées afin de documenter les transformations et les problèmes de qualité.

In [ ]:
RAW_DETAIL = ROOT / "data/raw/infractions_detail.csv"
RAW_TOTALS = ROOT / "data/raw/infractions_totaux_generaux.csv"
CLEAN_DATA = ROOT / "data/processed/infractions_clean.csv"
RECONCILIATION = ROOT / "data/processed/reconciliation_totaux.csv"

raw = pd.read_csv(
    RAW_DETAIL,
    sep=";",
    encoding="utf-8-sig",
    engine="python",
    dtype=object,
)

official_totals = pd.read_csv(
    RAW_TOTALS,
    sep=";",
    encoding="utf-8-sig",
    engine="python",
)

df = pd.read_csv(CLEAN_DATA)
reconciliation = pd.read_csv(RECONCILIATION)

print("Données brutes :", raw.shape)
print("Données nettoyées :", df.shape)
display(df.head())

## 3. Audit général de la base

Cette étape répond à la nécessité de décrire le volume, la couverture temporelle, les catégories et la granularité des observations.

In [ ]:
audit = pd.Series({
    "Nombre d'observations": len(df),
    "Nombre de variables": df.shape[1],
    "Nombre de périodes": df["Annee"].nunique(),
    "Nombre de catégories": df["Categorie"].nunique(),
    "Nombre d'infractions normalisées": df["Infraction_normalisee"].nunique(),
    "Libellés d'infraction non renseignés": int(df["Infraction_non_renseignee"].sum()),
    "Lignes sans activité": int((df["Volume_activite"] == 0).sum()),
    "Doublons Record_ID": int(df["Record_ID"].duplicated().sum()),
})
display(audit.to_frame("Valeur"))

print("\nRépartition par période :")
display(df["Annee"].value_counts().sort_index().to_frame("n"))

print("\nRépartition par catégorie :")
display(df["Categorie"].value_counts().to_frame("n"))

## 4. Valeurs manquantes

Il faut distinguer les données réellement absentes des zéros qui correspondent à une valeur numérique nulle.

La variable `Observations_Objets_saisis` est essentiellement textuelle et son absence n'est pas traitée comme une absence d'information numérique.

In [ ]:
missing_raw = (
    raw.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("Nombre manquant")
    .to_frame()
)
missing_raw["Pourcentage"] = 100 * missing_raw["Nombre manquant"] / len(raw)

print("Valeurs manquantes dans le fichier brut :")
display(missing_raw[missing_raw["Nombre manquant"] > 0].round(2))

missing_clean = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("Nombre manquant")
    .to_frame()
)
missing_clean["Pourcentage"] = 100 * missing_clean["Nombre manquant"] / len(df)

print("\nValeurs manquantes après nettoyage :")
display(missing_clean[missing_clean["Nombre manquant"] > 0].round(2))

## 5. Réconciliation avec les totaux officiels

Cette étape est essentielle : elle vérifie si la somme des lignes détaillées correspond aux totaux généraux officiellement fournis.

**Un écart n'est pas automatiquement une erreur du programme.** Il peut provenir d'une incohérence du fichier source, d'un sous-total mal saisi, d'une catégorie manquante ou d'une différence de définition statistique. Les écarts doivent être signalés dans le mémoire et, si possible, vérifiés auprès de la source métier.

In [ ]:
reconciliation["Concordance"] = reconciliation["Concordance"].astype(bool)

n_total = len(reconciliation)
n_diff = int((~reconciliation["Concordance"]).sum())

print(f"Comparaisons effectuées : {n_total}")
print(f"Écarts détectés : {n_diff}")
print(f"Taux de concordance : {(n_total - n_diff) / n_total:.1%}")

differences = reconciliation.loc[~reconciliation["Concordance"]].copy()
differences["Ecart_absolu"] = differences["Ecart"].abs()

display(
    differences
    .sort_values("Ecart_absolu", ascending=False)
    .drop(columns="Ecart_absolu")
)

print("\nMatrice des écarts par indicateur et période :")
display(
    reconciliation
    .pivot(index="Indicateur", columns="Annee", values="Ecart")
    .fillna(0)
)

### Interprétation à mettre dans le mémoire

Si des écarts subsistent, écrire explicitement que :

- les données détaillées ne reproduisent pas parfaitement tous les totaux officiels ;
- les analyses sont réalisées sur la table détaillée disponible ;
- les incohérences ont été identifiées et quantifiées ;
- les résultats doivent donc être interprétés comme exploratoires ;
- une validation métier des écarts est recommandée avant tout usage opérationnel.

## 6. Statistiques descriptives et asymétrie

Pour des données de comptage, la moyenne seule peut être trompeuse. On compare donc moyenne, médiane, dispersion, maximum, asymétrie et proportion de zéros.

In [ ]:
NUMERIC = [
    "Saisine_Plaintes_directes",
    "Saisine_FD",
    "Saisine_ST",
    "Saisine_Autres",
    "Victimes_H",
    "Victimes_F",
    "Victimes_G",
    "Victimes_F_fille",
    "MiseEnCause_Neutralises",
    "MiseEnCause_Interpelles",
    "Resultats_En_cours",
    "Resultats_Retraits",
    "Resultats_DAT",
    "Resultats_DEF",
    "Total_saisines",
    "Total_victimes",
    "Total_resultats",
    "Volume_activite",
]

stats = df[NUMERIC].describe().T
stats["mediane"] = df[NUMERIC].median()
stats["asymetrie_skewness"] = df[NUMERIC].skew()
stats["pourcentage_zero"] = 100 * df[NUMERIC].eq(0).mean()

display(
    stats[
        ["mean", "mediane", "std", "min", "25%", "50%", "75%", "max",
         "asymetrie_skewness", "pourcentage_zero"]
    ].round(2)
)

### Lecture scientifique

- Une **moyenne très supérieure à la médiane** indique généralement une distribution tirée vers le haut par quelques valeurs importantes.
- Une **skewness positive élevée** confirme une forte asymétrie à droite.
- Une forte proportion de zéros signifie que plusieurs variables sont **zero-inflated** ou très rares.
- Ces caractéristiques justifient l'usage de méthodes robustes et la prudence avec les algorithmes sensibles aux distances euclidiennes.

## 7. Distribution des variables principales

In [ ]:
variables_a_visualiser = [
    "Total_saisines",
    "Total_victimes",
    "MiseEnCause_Interpelles",
    "Resultats_DEF",
]

for var in variables_a_visualiser:
    plt.figure(figsize=(7, 4))
    plt.hist(df[var], bins=25)
    plt.title(f"Distribution de {var}")
    plt.xlabel(var)
    plt.ylabel("Fréquence")
    plt.tight_layout()
    plt.show()

### Variante logarithmique pour mieux voir la structure

La transformation `log1p(x)` ne remplace pas les valeurs originales. Elle sert uniquement à visualiser des distributions très asymétriques tout en conservant les zéros.

In [ ]:
for var in variables_a_visualiser:
    plt.figure(figsize=(7, 4))
    plt.hist(np.log1p(df[var]), bins=25)
    plt.title(f"Distribution logarithmique de {var}")
    plt.xlabel(f"log(1 + {var})")
    plt.ylabel("Fréquence")
    plt.tight_layout()
    plt.show()

## 8. Corrélations

La corrélation de Spearman est utilisée ici car les variables sont fortement asymétriques et contiennent de nombreux zéros.

**Attention : corrélation ne signifie pas causalité.**

In [ ]:
corr = df[NUMERIC].corr(method="spearman")

plt.figure(figsize=(14, 11))
im = plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(im, label="Corrélation de Spearman")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Matrice de corrélation de Spearman")
plt.tight_layout()
plt.show()

In [ ]:
pairs = []
cols = corr.columns.tolist()

for i, c1 in enumerate(cols):
    for j in range(i + 1, len(cols)):
        c2 = cols[j]
        r = corr.loc[c1, c2]
        pairs.append((c1, c2, r, abs(r)))

top_corr = (
    pd.DataFrame(pairs, columns=["Variable_1", "Variable_2", "rho", "abs_rho"])
    .sort_values("abs_rho", ascending=False)
)

display(top_corr.head(20).round(3))

### Interprétation

Les corrélations élevées entre variables de volume peuvent indiquer :

- une relation statistique réelle entre les phénomènes ;
- une redondance entre variables ;
- un risque de multicolinéarité pour les modèles linéaires.

Ridge est justement intéressant car sa régularisation réduit l'instabilité liée à la multicolinéarité.

## 9. Valeurs aberrantes selon la règle IQR

Les valeurs atypiques ne doivent pas être supprimées automatiquement. Dans des données criminelles, une valeur élevée peut représenter un événement réel et important.

In [ ]:
def iqr_outlier_summary(data, columns, factor=1.5):
    rows = []
    for col in columns:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        low = q1 - factor * iqr
        high = q3 + factor * iqr
        mask = (data[col] < low) | (data[col] > high)
        rows.append({
            "variable": col,
            "borne_basse": low,
            "borne_haute": high,
            "nb_atypiques": int(mask.sum()),
            "pourcentage": 100 * mask.mean(),
        })
    return pd.DataFrame(rows)

iqr_report = iqr_outlier_summary(df, NUMERIC)
display(iqr_report.sort_values("nb_atypiques", ascending=False).round(2))

In [ ]:
top_extremes = df.nlargest(
    15, "Volume_activite"
)[[
    "Annee",
    "Categorie",
    "Infraction_Affaire_traitee",
    "Total_saisines",
    "Total_victimes",
    "MiseEnCause_Interpelles",
    "Resultats_DEF",
    "Volume_activite",
]]

display(top_extremes)

## 10. Comparaison des périodes

**2026-S1 ne couvre qu'un semestre.** Il ne faut donc pas comparer directement ses totaux bruts à ceux de 2024 et 2025 sans le signaler.

Une annualisation simple peut être montrée à titre exploratoire, mais elle suppose implicitement que le second semestre suivrait le même rythme, ce qui n'est pas garanti.

In [ ]:
period_totals = df.groupby("Annee")[
    ["Total_saisines", "Total_victimes", "MiseEnCause_Interpelles", "Resultats_DEF"]
].sum()

display(period_totals)

annualized = period_totals.copy()
if "2026-S1" in annualized.index:
    annualized.loc["2026-S1"] = annualized.loc["2026-S1"] * 2

print("Annualisation exploratoire de 2026-S1 :")
display(annualized)

## 11. Préparation pour la régression

Cible : `Resultats_DEF`

Variables explicatives :

- période ;
- catégorie ;
- types de saisine ;
- victimes ;
- personnes neutralisées ;
- personnes interpellées.

Les variables directement dérivées de `Resultats_DEF`, comme `Total_resultats`, ne sont pas utilisées comme prédicteurs afin d'éviter une fuite d'information.

In [ ]:
TARGET = "Resultats_DEF"

CATEGORICAL_FEATURES = ["Annee", "Categorie"]

NUMERIC_FEATURES = [
    "Saisine_Plaintes_directes",
    "Saisine_FD",
    "Saisine_ST",
    "Saisine_Autres",
    "Victimes_H",
    "Victimes_F",
    "Victimes_G",
    "Victimes_F_fille",
    "MiseEnCause_Neutralises",
    "MiseEnCause_Interpelles",
]

FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

X = df[FEATURES].copy()
y = df[TARGET].astype(float)

def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_encoder()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
])

## 12. Comparaison de plusieurs modèles avec validation croisée

La validation croisée KFold donne une estimation plus robuste qu'une seule séparation entraînement/test sur un échantillon aussi petit.

Les modèles sont comparés à une **référence médiane**, indispensable pour vérifier que le Machine Learning apporte réellement quelque chose.

In [ ]:
models = {
    "Référence médiane": DummyRegressor(strategy="median"),
    "Ridge": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=350,
        max_depth=8,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=220,
        learning_rate=0.035,
        max_depth=3,
        loss="huber",
        random_state=42,
    ),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rows = []

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", estimator),
    ])

    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error",
            "r2": "r2",
        },
        n_jobs=-1,
    )

    rows.append({
        "Modele": name,
        "MAE_CV": -scores["test_mae"].mean(),
        "RMSE_CV": np.sqrt(-scores["test_mse"].mean()),
        "R2_CV": scores["test_r2"].mean(),
        "R2_ecart_type": scores["test_r2"].std(),
    })

results_cv = pd.DataFrame(rows).sort_values("RMSE_CV")
display(results_cv.round(4))

### Interprétation des métriques

- **MAE** : erreur absolue moyenne. Facile à interpréter dans l'unité de la cible.
- **RMSE** : pénalise davantage les grosses erreurs ; utile lorsqu'on veut éviter des écarts importants.
- **R²** : proportion de variabilité expliquée par le modèle. Un R² négatif indique un modèle moins bon qu'une prédiction naïve fondée sur une valeur constante.
- **Écart-type du R²** : mesure la stabilité du modèle selon les plis de validation.

Le meilleur modèle dépend du critère retenu. Il faut donc annoncer explicitement le critère de sélection.

## 13. Validation par période

La validation KFold mélange les périodes. Pour tester la robustesse temporelle, on réalise aussi une validation où une période entière est laissée de côté.

Cette analyse est particulièrement importante pour éviter de présenter un modèle exploratoire comme un vrai modèle prédictif temporel.

In [ ]:
group_cv = GroupKFold(n_splits=df["Annee"].nunique())

group_rows = []

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocess", preprocessor),
        ("model", estimator),
    ])

    scores = cross_validate(
        pipe,
        X,
        y,
        groups=df["Annee"],
        cv=group_cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error",
            "r2": "r2",
        },
        n_jobs=-1,
    )

    group_rows.append({
        "Modele": name,
        "MAE_par_periode": -scores["test_mae"].mean(),
        "RMSE_par_periode": np.sqrt(-scores["test_mse"].mean()),
        "R2_par_periode": scores["test_r2"].mean(),
        "R2_ecart_type": scores["test_r2"].std(),
    })

results_group = pd.DataFrame(group_rows).sort_values("RMSE_par_periode")
display(results_group.round(4))

### Pourquoi présenter les deux validations ?

- **KFold aléatoire** : mesure la performance interne sur des observations de structure similaire.
- **Validation par période** : mesure mieux la capacité du modèle à rester performant lorsqu'une période complète n'est pas observée pendant l'entraînement.

Si les classements des modèles changent, cela montre que le choix du modèle dépend du scénario de validation. C'est un résultat scientifique important.

## 14. Prédictions croisées et résidus du meilleur modèle KFold

In [ ]:
best_name = results_cv.iloc[0]["Modele"]
best_estimator = models[best_name]

best_pipe = Pipeline([
    ("preprocess", preprocessor),
    ("model", best_estimator),
])

pred_cv = cross_val_predict(best_pipe, X, y, cv=cv, n_jobs=-1)
pred_cv = np.maximum(pred_cv, 0)

mae = mean_absolute_error(y, pred_cv)
rmse = mean_squared_error(y, pred_cv) ** 0.5
r2 = r2_score(y, pred_cv)

print("Meilleur modèle KFold :", best_name)
print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y, pred_cv)
limit = max(y.max(), pred_cv.max())
plt.plot([0, limit], [0, limit], linestyle="--")
plt.xlabel("Valeur réelle")
plt.ylabel("Prédiction en validation croisée")
plt.title(f"Valeurs réelles vs prédites — {best_name}")
plt.tight_layout()
plt.show()

In [ ]:
residuals = y.to_numpy() - pred_cv

plt.figure(figsize=(7, 4))
plt.scatter(pred_cv, residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Prédiction")
plt.ylabel("Résidu = réel - prédit")
plt.title(f"Analyse des résidus — {best_name}")
plt.tight_layout()
plt.show()

### Discussion des résidus

À commenter dans le mémoire :

- présence éventuelle de très grosses erreurs ;
- tendance des erreurs à augmenter avec le volume ;
- sous-estimation ou surestimation systématique des valeurs élevées ;
- conséquence de la forte asymétrie des données.

Une bonne valeur moyenne de R² ne garantit pas que les cas extrêmes soient bien prédits.

## 15. Importance exploratoire des variables

Cette importance est calculée après ajustement du meilleur pipeline sur l'ensemble des données. Elle doit être décrite comme **exploratoire**, pas comme une preuve causale.

In [ ]:
best_pipe.fit(X, y)

importance = permutation_importance(
    best_pipe,
    X,
    y,
    n_repeats=20,
    random_state=42,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "Variable": FEATURES,
    "Importance": importance.importances_mean,
}).sort_values("Importance", ascending=False)

display(importance_df.round(4))

In [ ]:
plt.figure(figsize=(8, 5))
plot_df = importance_df.sort_values("Importance")
plt.barh(plot_df["Variable"], plot_df["Importance"])
plt.xlabel("Diminution de performance par permutation")
plt.title("Importance exploratoire des variables")
plt.tight_layout()
plt.show()

## 16. Clustering : vérification de robustesse

Dans la version actuelle du projet, K-Means est appliqué à des variables de comptage fortement asymétriques.

On compare donc :

1. standardisation directe des valeurs brutes ;
2. transformation `log1p` puis standardisation.

Cette vérification permet de savoir si le clustering sépare réellement plusieurs profils ou s'il isole surtout quelques observations extrêmement volumineuses.

In [ ]:
CLUSTER_FEATURES = [
    "Total_saisines",
    "Total_victimes",
    "MiseEnCause_Interpelles",
    "MiseEnCause_Neutralises",
    "Resultats_DEF",
    "Resultats_En_cours",
]

def clustering_sensitivity(data, transform):
    Xc = data[CLUSTER_FEATURES].astype(float).to_numpy()

    if transform == "log1p":
        Xc = np.log1p(Xc)

    Xc = StandardScaler().fit_transform(Xc)

    rows = []
    for k in range(2, 7):
        labels = KMeans(
            n_clusters=k,
            n_init=30,
            random_state=42,
        ).fit_predict(Xc)

        counts = pd.Series(labels).value_counts().sort_index().to_dict()

        rows.append({
            "Transformation": transform,
            "k": k,
            "Silhouette": silhouette_score(Xc, labels),
            "Tailles_clusters": counts,
        })

    return pd.DataFrame(rows)

cluster_raw = clustering_sensitivity(df, "brut")
cluster_log = clustering_sensitivity(df, "log1p")

display(pd.concat([cluster_raw, cluster_log], ignore_index=True))

### Interprétation du clustering

Un score silhouette très élevé n'est pas toujours synonyme d'une segmentation scientifiquement utile.

Si `k=2` produit par exemple un groupe immense et un groupe de quelques observations seulement, le modèle peut surtout être en train d'isoler des valeurs extrêmes.

Une transformation logarithmique peut révéler une structure plus équilibrée. Cette comparaison doit être discutée avant de donner un sens métier aux clusters.

## 17. Visualisation PCA du clustering log-transformé

Cette visualisation est exploratoire. La PCA projette les données sur deux axes et ne conserve pas toute l'information.

In [ ]:
X_cluster = np.log1p(df[CLUSTER_FEATURES].astype(float))
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

k = 3
kmeans = KMeans(n_clusters=k, n_init=30, random_state=42)
labels = kmeans.fit_predict(X_cluster_scaled)

pca = PCA(n_components=2)
coords = pca.fit_transform(X_cluster_scaled)

print("Variance expliquée par les deux axes :", pca.explained_variance_ratio_.sum())

plt.figure(figsize=(7, 5))
for cluster_id in sorted(np.unique(labels)):
    mask = labels == cluster_id
    plt.scatter(
        coords[mask, 0],
        coords[mask, 1],
        label=f"Cluster {cluster_id}",
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Clustering après transformation log1p")
plt.legend()
plt.tight_layout()
plt.show()

## 18. Détection d'anomalies : analyse de sensibilité

Le paramètre `contamination` d'Isolation Forest fixe approximativement la proportion d'observations considérées comme anormales.

Il doit donc être justifié et testé à plusieurs valeurs.

In [ ]:
ANOMALY_FEATURES = [
    "Saisine_Plaintes_directes",
    "Saisine_FD",
    "Saisine_ST",
    "Saisine_Autres",
    "Total_victimes",
    "MiseEnCause_Neutralises",
    "MiseEnCause_Interpelles",
    "Resultats_En_cours",
    "Resultats_Retraits",
    "Resultats_DAT",
    "Resultats_DEF",
]

Xa = df[ANOMALY_FEATURES].astype(float)

sensitivity = []

for contamination in [0.03, 0.05, 0.08, 0.10, 0.15]:
    pipeline = Pipeline([
        ("scaler", RobustScaler()),
        ("model", IsolationForest(
            n_estimators=400,
            contamination=contamination,
            random_state=42,
            n_jobs=-1,
        )),
    ])

    pred = pipeline.fit_predict(Xa)

    sensitivity.append({
        "Contamination": contamination,
        "Nombre_anomalies_ML": int((pred == -1).sum()),
        "Pourcentage": 100 * (pred == -1).mean(),
    })

display(pd.DataFrame(sensitivity).round(2))

### Interprétation des anomalies

Une anomalie statistique peut être :

- une erreur de saisie ;
- une valeur rare mais correcte ;
- un événement exceptionnel ;
- une conséquence de l'agrégation des données.

Il ne faut donc pas supprimer automatiquement les anomalies. Elles doivent être vérifiées avec les documents sources ou par un expert métier.

## 19. Synthèse automatique des principaux constats

In [ ]:
n = len(df)
periods = ", ".join(sorted(df["Annee"].unique()))
n_diff = int((~reconciliation["Concordance"]).sum())

skews = df[[
    "Total_saisines",
    "Total_victimes",
    "MiseEnCause_Interpelles",
    "Resultats_DEF",
]].skew()

print("SYNTHÈSE SCIENTIFIQUE")
print("-" * 60)
print(f"• {n} observations couvrant : {periods}.")
print(f"• {n_diff} écarts ont été détectés entre détail et totaux officiels.")
print("• Les principales variables de comptage présentent une asymétrie positive importante.")
print("• La comparaison des modèles doit être interprétée relativement à une baseline.")
print("• La validation par période complète l'évaluation KFold aléatoire.")
print("• Le clustering brut doit être interprété prudemment car les valeurs extrêmes influencent K-Means.")
print("• Le nombre d'anomalies dépend du paramètre de contamination.")
print("• Les résultats restent exploratoires en raison du faible volume, de l'agrégation et de l'absence de géolocalisation.")
print("\nAsymétrie des variables principales :")
display(skews.to_frame("Skewness").round(3))

## 20. Discussion critique à reprendre dans le mémoire

### Forces

- Pipeline Data Science complet et reproductible.
- Nettoyage automatique et contrôle de qualité.
- Comparaison de plusieurs modèles avec baseline.
- Validation croisée.
- Clustering, PCA et détection d'anomalies.
- API REST, base de données et interface web fonctionnelles.
- Tests automatisés.

### Limites

- Échantillon de seulement 152 observations.
- Données agrégées par infraction et période.
- Seulement deux années complètes et un semestre.
- Absence de données mensuelles ou journalières.
- Absence de quartier, adresse, latitude et longitude.
- Plusieurs écarts entre détails et totaux officiels.
- Forte asymétrie et nombreuses valeurs nulles.
- Les clusters peuvent être influencés par quelques observations extrêmes.
- Le réglage d'Isolation Forest est sensible au paramètre `contamination`.
- La régression est une preuve de concept et ne constitue pas une prédiction spatiale ou temporelle opérationnelle.

### Perspectives

- Collecter des données géolocalisées et horodatées.
- Ajouter des variables contextuelles : population, densité, événements, météo, mobilité, caractéristiques socio-économiques.
- Exploiter les séries temporelles.
- Intégrer un SIG.
- Tester des méthodes de comptage adaptées et des modèles temporels lorsque le volume de données sera suffisant.
- Ajouter des méthodes d'Explainable AI, par exemple SHAP, sur un jeu de données suffisamment robuste.
- Réaliser une validation métier avec des analystes, collectivités ou forces de l'ordre.

## Conclusion

Le notebook ne transforme pas le projet en système de prédiction opérationnelle. Son objectif est de renforcer la **rigueur scientifique** du mémoire : vérifier les données, justifier les choix, comparer les modèles, interpréter les métriques et discuter clairement les limites.

C'est cette distinction entre **réalisation technique** et **démonstration scientifique** qui doit apparaître dans la version finale du mémoire.